# 实验 1：认识 Qwen3-0.6B 的外部骨架

## 今天只回答一个问题

**一个 token 进入 Qwen3-0.6B 之后，要依次经过哪些主要模块？**

本实验只观察模型的**顶层流水线**和**模块树**：

1. 先看一张总览图，建立整体地图；
2. 再从真实模型对象自动生成一棵精简树；
3. 最后把图和树放在一起，确认它们描述的是同一副骨架。

今天暂时不打开 Transformer Block 的内部。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。模型会加载约 1.2GB 权重，但本实验不生成文本、不训练模型。

In [8]:
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'模型目录: {MODEL_PATH}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')


项目根目录: /home/linjunjie/Workspace/xxdw1
模型目录: /home/linjunjie/Workspace/xxdw1/models/Qwen3-0.6B-Base
PyTorch: 2.13.0+cu130
Transformers: 5.15.1


In [9]:
# 这里只加载模型，不做生成；设备固定为 CPU，方便专注观察结构。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).cpu().eval()

print(f'模型类: {type(model).__name__}')
print(f'模型设备: {next(model.parameters()).device}')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型类: Qwen3ForCausalLM
模型设备: cpu


## 1. 先看模型的总览图

今天只观察模型的**外部流水线**，先不打开 Transformer Block 的内部。

<div align="center">
  <img src="../assets/qwen3-backbone-overview.png" alt="Qwen3 顶层结构总览图" width="360" style="max-width: 100%; border: 1px solid #cbd5e1; border-radius: 8px;">
</div>

先把 Block 当作黑盒：它接收 hidden states，处理后交给下一阶段。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。

## 2. 从真实模型对象生成精简模块树

下面的树不是手写示意图。代码会读取刚刚加载的 `model` 对象，自动取得模块类名、层数和维度。

阅读缩进的方法：

- `model` 和 `lm_head` 是 `Qwen3ForCausalLM` 直接包含的两个顶层部分；
- `embed_tokens`、`layers`、`norm` 都位于内部的 `Qwen3Model` 中；
- `layers × 28` 暂时作为一组黑盒，不在本实验继续展开。

In [10]:
config = model.config
backbone = model.model
embedding = backbone.embed_tokens
blocks = backbone.layers
final_norm = backbone.norm
lm_head = model.lm_head

print(type(model).__name__)
print(f"├── model: {type(backbone).__name__}")
print(
    f"│   ├── embed_tokens: {type(embedding).__name__}"
    f"({embedding.num_embeddings:,}, {embedding.embedding_dim})"
)
print(
    f"│   ├── layers: {type(blocks).__name__} × {len(blocks)} "
    f"[{type(blocks[0]).__name__}]"
)
print(f"│   └── norm: {type(final_norm).__name__}({config.hidden_size})")
print(
    f"└── lm_head: {type(lm_head).__name__}"
    f"({lm_head.in_features}, {lm_head.out_features}, bias={lm_head.bias is not None})"
)


Qwen3ForCausalLM
├── model: Qwen3Model
│   ├── embed_tokens: Embedding(151,936, 1024)
│   ├── layers: ModuleList × 28 [Qwen3DecoderLayer]
│   └── norm: Qwen3RMSNorm(1024)
└── lm_head: Linear(1024, 151936, bias=False)


## 3. 小结：今天只记住这副外部骨架

```text
Input Token
    ↓
Embedding                model.model.embed_tokens
    ↓
28 × Transformer Block   model.model.layers
    ↓
Final RMSNorm            model.model.norm
    ↓
LM Head                  model.lm_head
    ↓
Output Logits
```

完成本实验后，你应该能够回答：

1. 一段 token 序列进入模型后，会依次经过哪些顶层阶段？
2. `Qwen3ForCausalLM`、`Qwen3Model`、`embed_tokens`、`layers`、`norm` 和 `lm_head` 在模块树中是什么包含关系？

本实验到这里停止。下一实验再选择一个黑盒打开观察。